# The aim is to prepare the beans dataset
https://github.com/earthspecies/beans/tree/main

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from tqdm.notebook import tqdm

In [3]:
from esp_data import AnyPath

In [4]:
import esp_data.file_io.functional as F

In [5]:
from pathlib import Path

In [6]:
import duckdb

In [7]:
dataset_names_and_types = {
    "cbi": "classification",
    "watkins": "classification",
    "humbugdb": "classification",
    "egyptian_fruit_bats": "classification",
    "dogs": "classification",
    "hiceas": "detection",
    "dcase": "detection",
    "enabirds": "detection",
    "rfcx": "detection",
    "hainan_gibbons": "detection",
    "esc50": "classification",
    "speech-commands": "classification"
}

In [8]:
paths_to_annotations = {
    "cbi": {"train": "gs://foundation-model-data/audio/cbi/annotations.train.csv",
            "test": "gs://foundation-model-data/audio/cbi/annotations.test.csv",
            "validation": "gs://foundation-model-data/audio/cbi/annotations.valid.csv",
           },
    "watkins": {"train": "gs://foundation-model-data/audio/watkins/annotations.train.csv",
                "test": "gs://foundation-model-data/audio/watkins/annotations.test.csv",
                "validation": "gs://foundation-model-data/audio/watkins/annotations.valid.csv",
               },
    "humbugdb": {
        "train": "gs://foundation-model-data/audio/HumBugDB/data/metadata/train.csv",
        "test": "gs://foundation-model-data/audio/HumBugDB/data/metadata/test.csv",
        "validation": "gs://foundation-model-data/audio/HumBugDB/data/metadata/valid.csv",
    },
    "egyptian_fruit_bats": {
        "train": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.train.csv",
        "test": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.test.csv",
        "validation": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.valid.csv",
    },
    "dogs": {
        "train": "gs://foundation-model-data/audio/dogs/annotations.train.csv",
        "test": "gs://foundation-model-data/audio/dogs/annotations.test.csv",
        "validation": "gs://foundation-model-data/audio/dogs/annotations.valid.csv",
    },
    "hiceas": {
        "train": "gs://foundation-model-data/data/hiceas-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/hiceas-detection/test-fixed.jsonl",
        "validation": "gs://foundation-model-data/data/hiceas-detection/valid.jsonl",
    },
    "dcase": {
        "train": "gs://foundation-model-data/data/dcase-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/dcase-detection/test-fixed.jsonl",
        "validation": "gs://foundation-model-data/data/dcase-detection/valid.jsonl",
    },
    "enabirds": {
        "train": "gs://foundation-model-data/data/enabirds-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/enabirds-detection/test-fixed.jsonl",
        "validation": "gs://foundation-model-data/data/enabirds-detection/valid.jsonl",
    },
    "rfcx": {
        "train": "gs://foundation-model-data/data/rfcx-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/rfcx-detection/test-fixed.jsonl",
        "validation": "gs://foundation-model-data/data/rfcx-detection/valid.jsonl",
    },
    "hainan_gibbons": {
        "train": "gs://foundation-model-data/data/hainan-gibbons-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/hainan-gibbons-detection/test-fixed.jsonl",
        "validation": "gs://foundation-model-data/data/hainan-gibbons-detection/valid.jsonl",
    },
    "esc50-all": {
        "train": "gs://foundation-model-data/data/esc50-all/train.jsonl",
        "test": "gs://foundation-model-data/data/esc50-all/test.jsonl",
        "validation": "gs://foundation-model-data/data/esc50-all/valid.jsonl"
    },
    "speech-commands": {
        "train": "gs://foundation-model-data/data/speech-commands-classification/train.jsonl",
        "test": "gs://foundation-model-data/data/speech-commands-classification/test.jsonl",
        "validation": "gs://foundation-model-data/data/speech-commands-classification/valid.jsonl"
    }
}

In [9]:
beans_root = AnyPath("gs://esp-ml-datasets/beans/v0.1.0/raw/")
beans_root

GSPath('gs://esp-ml-datasets/beans/v0.1.0/raw/')

In [10]:
train_dfs = {}
test_dfs = {}
validation_dfs = {}

for dsname, splits in paths_to_annotations.items():

    for s,v in splits.items():

        if "jsonl" in v:
            df = pd.read_json(v, lines=True, orient="records")
        elif "csv" in v:
            df = pd.read_csv(v)

        if s == "train":
            train_dfs[dsname] = df
        elif s == "test":
            test_dfs[dsname] = df
        else:
            validation_dfs[dsname] = df

In [11]:
train_cols = [list(v.columns) for k,v in train_dfs.items()]

In [12]:
paths_to_annotations.keys()

dict_keys(['cbi', 'watkins', 'humbugdb', 'egyptian_fruit_bats', 'dogs', 'hiceas', 'dcase', 'enabirds', 'rfcx', 'hainan_gibbons', 'esc50-all', 'speech-commands'])

In [13]:
train_cols

[['Unnamed: 0', 'path', 'label', 'split'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'label'],
 ['instruction', 'path', 'label']]

# Fix all

## CBI

In [14]:
train_dfs["cbi"].head(5)

,Unnamed: 0,path,label,split
0,1,data/cbi/wav/XC135454.wav,aldfly,train
1,2,data/cbi/wav/XC135455.wav,aldfly,train
2,3,data/cbi/wav/XC135456.wav,aldfly,train
3,4,data/cbi/wav/XC135457.wav,aldfly,train
4,5,data/cbi/wav/XC135459.wav,aldfly,train


### create file_name and local_path cols

In [15]:
train_dfs["cbi"]["file_name"] = train_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
train_dfs["cbi"]["local_path"] = train_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)

In [16]:
test_dfs["cbi"]["file_name"] = test_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
test_dfs["cbi"]["local_path"] = test_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)
validation_dfs["cbi"]["file_name"] = validation_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
validation_dfs["cbi"]["local_path"] = validation_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)

### drop split column and Unnamed:0

In [17]:
train_dfs["cbi"] = train_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)

In [18]:
test_dfs["cbi"] = test_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)
validation_dfs["cbi"] = validation_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)

In [19]:
train_dfs["cbi"].columns

Index(['label', 'file_name', 'local_path'], dtype='object')

In [20]:
print(test_dfs["cbi"], validation_dfs["cbi"])

       label     file_name                  local_path
0     aldfly  XC137570.wav  audio/cbi/wav/XC137570.wav
1     aldfly  XC154310.wav  audio/cbi/wav/XC154310.wav
2     aldfly  XC157462.wav  audio/cbi/wav/XC157462.wav
3     aldfly  XC180091.wav  audio/cbi/wav/XC180091.wav
4     aldfly  XC182735.wav  audio/cbi/wav/XC182735.wav
...      ...           ...                         ...
3615  yetvir  XC376747.wav  audio/cbi/wav/XC376747.wav
3616  yetvir  XC417474.wav  audio/cbi/wav/XC417474.wav
3617  yetvir  XC421946.wav  audio/cbi/wav/XC421946.wav
3618  yetvir  XC467630.wav  audio/cbi/wav/XC467630.wav
3619  yetvir  XC471316.wav  audio/cbi/wav/XC471316.wav

[3620 rows x 3 columns]        label     file_name                  local_path
0     aldfly  XC134874.wav  audio/cbi/wav/XC134874.wav
1     aldfly  XC135883.wav  audio/cbi/wav/XC135883.wav
2     aldfly  XC138639.wav  audio/cbi/wav/XC138639.wav
3     aldfly  XC139577.wav  audio/cbi/wav/XC139577.wav
4     aldfly  XC154449.wav  audio/cbi/wa

### Perform a check that every audio file exists

In [21]:
dsname = "cbi"

def check_audio_exists(dsname, extensions: list[str] = ["wav", "flac", "mp3"]):
    audio_root = beans_root / f"audio/{dsname}" 
    all_audios = []
    for e in extensions:
        all_audios.extend(F.list_files(audio_root, f"**/*.{e}", use_fs=True))
    print(f"Found {len(all_audios)} audio files for {dsname}")
    
    for split, df in zip(["train", "test", "val"], [train_dfs[dsname], test_dfs[dsname], validation_dfs[dsname]]):
        files_in_wavs = df["local_path"].apply(lambda x: str(beans_root / x)).isin(all_audios).sum()
        assert files_in_wavs == len(df), f"Woops! Mismatch for {split} in {dsname}"

## Watkins

In [22]:
train_dfs["watkins"].sample(5)

,Unnamed: 0,path,label
446,747,"data/watkins/Grampus,_Rissos_Dolphin/85009005.wav","Grampus,_Rissos_Dolphin"
727,1212,data/watkins/Bowhead_Whale/7801800J.wav,Bowhead_Whale
377,629,data/watkins/Frasers_Dolphin/9101300U.wav,Frasers_Dolphin
232,383,data/watkins/Southern_Right_Whale/79002007.wav,Southern_Right_Whale
832,1395,data/watkins/Long-Finned_Pilot_Whale/7703400H.wav,Long-Finned_Pilot_Whale


In [23]:
train_dfs["watkins"]["file_name"] = train_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
train_dfs["watkins"]["local_path"] = train_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

In [24]:
test_dfs["watkins"]["file_name"] = test_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
test_dfs["watkins"]["local_path"] = test_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

validation_dfs["watkins"]["file_name"] = validation_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
validation_dfs["watkins"]["local_path"] = validation_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

In [25]:
train_dfs["watkins"] = train_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)

In [26]:

test_dfs["watkins"] = test_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)
validation_dfs["watkins"] = validation_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)
test_dfs["watkins"].columns

Index(['label', 'file_name', 'local_path'], dtype='object')

In [27]:
validation_dfs["watkins"].head(5)

,label,file_name,local_path
0,Clymene_Dolphin,8300604P.wav,audio/watkins/8300604P.wav
1,Clymene_Dolphin,83006031.wav,audio/watkins/83006031.wav
2,Clymene_Dolphin,83006022.wav,audio/watkins/83006022.wav
3,Clymene_Dolphin,8303503O.wav,audio/watkins/8303503O.wav
4,Clymene_Dolphin,8300600B.wav,audio/watkins/8300600B.wav


In [28]:
train_dfs["watkins"]["label"].unique()

array(['Clymene_Dolphin', 'Bottlenose_Dolphin', 'Spinner_Dolphin',
       'Beluga,_White_Whale', 'Bearded_Seal', 'Minke_Whale',
       'Humpback_Whale', 'Southern_Right_Whale', 'White-sided_Dolphin',
       'Narwhal', 'White-beaked_Dolphin', 'Northern_Right_Whale',
       'Frasers_Dolphin', 'Grampus,_Rissos_Dolphin', 'Harp_Seal',
       'Atlantic_Spotted_Dolphin', 'Fin,_Finback_Whale', 'Ross_Seal',
       'Rough-Toothed_Dolphin', 'Killer_Whale',
       'Pantropical_Spotted_Dolphin', 'Short-Finned_Pacific_Pilot_Whale',
       'Bowhead_Whale', 'False_Killer_Whale', 'Melon_Headed_Whale',
       'Long-Finned_Pilot_Whale', 'Striped_Dolphin', 'Leopard_Seal',
       'Walrus', 'Sperm_Whale', 'Common_Dolphin'], dtype=object)

In [ ]:
check_audio_exists("watkins", extensions=["wav"])

In [29]:

def do_process(dsname, cols_to_drop=["Unnamed: 0", "path"]):
    
    train_dfs[dsname]["file_name"] = train_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    train_dfs[dsname]["local_path"] = train_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)
    
    test_dfs[dsname]["file_name"] = test_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    test_dfs[dsname]["local_path"] = test_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)
    
    validation_dfs[dsname]["file_name"] = validation_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    validation_dfs[dsname]["local_path"] = validation_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)

    train_dfs[dsname] = train_dfs[dsname].drop(cols_to_drop, axis=1)
    test_dfs[dsname] = test_dfs[dsname].drop(cols_to_drop, axis=1)
    validation_dfs[dsname] = validation_dfs[dsname].drop(cols_to_drop, axis=1)
    

## Humbugdb

In [30]:
train_dfs["humbugdb"].sample(5, replace=False)

,Unnamed: 0,path,label
2236,3676,data/HumBugDB/data/audio/221720.wav,an squamosus
3172,5233,data/HumBugDB/data/audio/219197.wav,an gambiae ss
1151,1908,data/HumBugDB/data/audio/219950.wav,non-mosquito
5117,8521,data/HumBugDB/data/audio/201745.wav,non-mosquito
2991,4936,data/HumBugDB/data/audio/219548.wav,an gambiae ss


In [31]:
do_process("humbugdb")

In [32]:
train_dfs["humbugdb"].sample(5, replace=False)

,label,file_name,local_path
4345,non-mosquito,2624.wav,audio/humbugdb/2624.wav
4347,non-mosquito,2629.wav,audio/humbugdb/2629.wav
2183,culex pipiens complex,221681.wav,audio/humbugdb/221681.wav
4639,non-mosquito,202037.wav,audio/humbugdb/202037.wav
5102,non-mosquito,201748.wav,audio/humbugdb/201748.wav


# Egyptian fruit bats

In [33]:
train_dfs["egyptian_fruit_bats"].sample(10)

,Unnamed: 0,path,label
699,1150,data/egyptian_fruit_bats/audio/76186.wav,230
3565,5982,data/egyptian_fruit_bats/audio/47923.wav,231
5431,8986,data/egyptian_fruit_bats/audio/84545.wav,210
4671,7759,data/egyptian_fruit_bats/audio/25453.wav,216
4461,7416,data/egyptian_fruit_bats/audio/25020.wav,226
3032,5139,data/egyptian_fruit_bats/audio/51557.wav,220
2278,3868,data/egyptian_fruit_bats/audio/75721.wav,211
5048,8377,data/egyptian_fruit_bats/audio/65330.wav,220
3132,5308,data/egyptian_fruit_bats/audio/76088.wav,216
2102,3552,data/egyptian_fruit_bats/audio/78418.wav,215


In [34]:
do_process("egyptian_fruit_bats")

In [35]:
train_dfs["egyptian_fruit_bats"].sample(5)

,label,file_name,local_path
2122,216,34241.wav,audio/egyptian_fruit_bats/34241.wav
2971,220,79259.wav,audio/egyptian_fruit_bats/79259.wav
2276,230,73012.wav,audio/egyptian_fruit_bats/73012.wav
4756,211,60044.wav,audio/egyptian_fruit_bats/60044.wav
3952,228,14244.wav,audio/egyptian_fruit_bats/14244.wav


## Dogs

In [36]:
train_dfs["dogs"].sample(10)

,Unnamed: 0,path,label
384,636,data/dogs/wav/Luke-C-3a.wav,Luke
315,526,data/dogs/wav/Keri-6-P-6e.wav,Keri
2,2,data/dogs/wav/Mac-2-P-2d.wav,Mac
181,305,data/dogs/wav/Roodie-1-P-1c.wav,Roodie
186,311,data/dogs/wav/Roodie-1-P-1h.wav,Roodie
298,501,data/dogs/wav/Keri3-A-3b.wav,Keri
106,176,data/dogs/wav/Siggy-4-C-4f.wav,Siggy
377,626,data/dogs/wav/Luke-C-3c.wav,Luke
256,430,data/dogs/wav/Roodie-8-A-7.wav,Roodie
302,507,data/dogs/wav/Keri-7-A-7b.wav,Keri


In [37]:
do_process("dogs")

In [38]:
train_dfs["dogs"].sample(5)

,label,file_name,local_path
338,Freid,Freid-A-6.wav,audio/dogs/Freid-A-6.wav
28,Mac,Mac-5-C-3a.wav,audio/dogs/Mac-5-C-3a.wav
408,Luke,Luke-5-C-5b.wav,audio/dogs/Luke-5-C-5b.wav
164,Zoe,Zoe-1-P-1a.wav,audio/dogs/Zoe-1-P-1a.wav
352,Luke,Luke-C-4c.wav,audio/dogs/Luke-C-4c.wav


## Hiceas

In [39]:
train_dfs["hiceas"].sample(10)

,instruction,path,answer
651,{hiceas-detection},audio/hiceas-detection/1705_20170901_165054_14...,None
1019,{hiceas-detection},audio/hiceas-detection/1705_20170902_032826_57...,None
3437,{hiceas-detection},audio/hiceas-detection/1705_20171022_214847_32...,None
3080,{hiceas-detection},audio/hiceas-detection/1705_20171009_193428_94...,None
4236,{hiceas-detection},audio/hiceas-detection/1705_20171101_033719_52...,None
2097,{hiceas-detection},audio/hiceas-detection/1705_20170927_213920_37...,None
171,{hiceas-detection},audio/hiceas-detection/1705_20170827_225635_54...,None
657,{hiceas-detection},audio/hiceas-detection/1705_20170901_165054_14...,None
3808,{hiceas-detection},audio/hiceas-detection/1705_20171026_000752_07...,None
1964,{hiceas-detection},audio/hiceas-detection/1705_20170927_010414_50...,None


In [40]:
train_dfs["hiceas"].answer.unique()

array(['None', "Minke Whale 'boing' call"], dtype=object)

In [41]:
do_process("hiceas", cols_to_drop=["instruction", "path"])

In [42]:
train_dfs["hiceas"].sample(10)

,answer,file_name,local_path
4344,Minke Whale 'boing' call,1705_20171103_001035_105_010.wav,audio/hiceas/1705_20171103_001035_105_010.wav
1571,None,1705_20170904_203642_225_009.wav,audio/hiceas/1705_20170904_203642_225_009.wav
4016,None,1705_20171028_201127_033_001.wav,audio/hiceas/1705_20171028_201127_033_001.wav
3831,None,1705_20171026_002952_078_003.wav,audio/hiceas/1705_20171026_002952_078_003.wav
4353,Minke Whale 'boing' call,1705_20171103_002035_105_008.wav,audio/hiceas/1705_20171103_002035_105_008.wav
4078,None,1705_20171028_214108_280_008.wav,audio/hiceas/1705_20171028_214108_280_008.wav
3715,None,1705_20171025_203957_142_008.wav,audio/hiceas/1705_20171025_203957_142_008.wav
4271,Minke Whale 'boing' call,1705_20171102_223235_105_003.wav,audio/hiceas/1705_20171102_223235_105_003.wav
869,None,1705_20170902_001922_379_000.wav,audio/hiceas/1705_20170902_001922_379_000.wav
2907,None,1705_20171002_001945_475_003.wav,audio/hiceas/1705_20171002_001945_475_003.wav


## Dcase

In [43]:
train_dfs["dcase"].sample(5)

,instruction,path,answer
24806,{dcase-detection},audio/dcase-detection/2015-10-14_23-59-59_unit...,None
26043,{dcase-detection},audio/dcase-detection/e1.013_024.wav,None
35946,{dcase-detection},audio/dcase-detection/n1.027_015.wav,None
30194,{dcase-detection},audio/dcase-detection/y1.027_045.wav,None
12062,{dcase-detection},audio/dcase-detection/2015-09-04_08-04-59_unit...,None


In [ ]:
train_dfs["dcase"].answer.unique()

In [44]:
do_process("dcase", ["instruction", "path"])

In [45]:
train_dfs["dcase"].sample(5)

,answer,file_name,local_path
31635,None,dcase_MK1.015_011.wav,audio/dcase/dcase_MK1.015_011.wav
26703,None,e1.024_035.wav,audio/dcase/e1.024_035.wav
9392,None,2015-09-04_08-04-59_unit03.023_011.wav,audio/dcase/2015-09-04_08-04-59_unit03.023_011...
36599,None,BUK1_20181011_001004.006_019.wav,audio/dcase/BUK1_20181011_001004.006_019.wav
9041,None,2015-09-04_08-04-59_unit03.017_014.wav,audio/dcase/2015-09-04_08-04-59_unit03.017_014...


## Enabirds

In [46]:
train_dfs["enabirds"].sample(5)

,instruction,path,answer
12669,{enabirds-detection},audio/enabirds-detection/Recording_4_Segment_1...,"Northern Cardinal, American Crow, Black-capped..."
5123,{enabirds-detection},audio/enabirds-detection/Recording_1_Segment_0...,"Wood Thrush, Eastern Towhee"
13555,{enabirds-detection},audio/enabirds-detection/Recording_4_Segment_2...,"Eastern Towhee, Tufted Titmouse, Common Yellow..."
1159,{enabirds-detection},audio/enabirds-detection/Recording_1_Segment_2...,"Brown-headed Cowbird, Hermit Thrush"
7830,{enabirds-detection},audio/enabirds-detection/Recording_1_Segment_2...,"American Crow, Ovenbird"


In [47]:
do_process("enabirds", ["instruction", "path"])

In [48]:
train_dfs["enabirds"].sample(5)

,answer,file_name,local_path
2511,"Red-eyed Vireo, Wood Thrush, Ovenbird, Kirtlan...",Recording_2_Segment_07.000_033.wav,audio/enabirds/Recording_2_Segment_07.000_033.wav
1753,"Black-and-white Warbler, Black-throated Green ...",Recording_1_Segment_30.000_042.wav,audio/enabirds/Recording_1_Segment_30.000_042.wav
13356,"Eastern Towhee, Northern Cardinal, American Crow",Recording_4_Segment_24.002_022.wav,audio/enabirds/Recording_4_Segment_24.002_022.wav
9657,"Black-and-white Warbler, American Crow",Recording_2_Segment_08.001_040.wav,audio/enabirds/Recording_2_Segment_08.001_040.wav
10411,American Crow,Recording_2_Segment_14.002_027.wav,audio/enabirds/Recording_2_Segment_14.002_027.wav


## Rfcx

In [49]:
train_dfs["rfcx"].sample(5)

,instruction,path,answer
13593,{rfcx-detection},audio/rfcx-detection/7071fd618_008.wav,None
27953,{rfcx-detection},audio/rfcx-detection/e5dc8bd3b_002.wav,None
29912,{rfcx-detection},audio/rfcx-detection/f5087f6e9_003.wav,None
6937,{rfcx-detection},audio/rfcx-detection/396598a81_007.wav,None
13351,{rfcx-detection},audio/rfcx-detection/6dc49f821_008.wav,None


In [50]:
do_process("rfcx", ["instruction", "path"])

In [51]:
train_dfs["rfcx"].sample(5)

,answer,file_name,local_path
24544,None,c99a2fe6e_003.wav,audio/rfcx/c99a2fe6e_003.wav
11081,None,5af2b2a0c_004.wav,audio/rfcx/5af2b2a0c_004.wav
15835,None,8250cac8d_006.wav,audio/rfcx/8250cac8d_006.wav
8034,None,43739e7ab_004.wav,audio/rfcx/43739e7ab_004.wav
21070,None,ae4a3b1e2_005.wav,audio/rfcx/ae4a3b1e2_005.wav


## Hainan gibbons

In [52]:
train_dfs["hainan_gibbons"].sample(5)

,instruction,path,answer
7839,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
11173,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
3820,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
16426,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3A_0+1_2016...,None
25849,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3BD_0+1_201...,None


In [53]:
do_process("hainan_gibbons", ["instruction", "path"])

In [54]:
train_dfs["hainan_gibbons"].sample(5)

,answer,file_name,local_path
6546,None,HGSM3AC_0+1_20160312_055400.195_021.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160312_0554...
26493,None,HGSM3BD_0+1_20160401_053700.339_016.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160401_0537...
7112,None,HGSM3AC_0+1_20160312_055400.255_007.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160312_0554...
27096,None,HGSM3BD_0+1_20160401_053700.402_010.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160401_0537...
1571,None,HGSM3AC_0+1_20160309_055600.162_005.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160309_0556...


## esc50-all

In [55]:
train_dfs["esc50-all"].sample(5)

,instruction,path,label
1092,{esc50-all},audio/esc50/3-166546-A-34.wav,can_opening
657,{esc50-all},audio/esc50/2-52789-A-4.wav,frog
899,{esc50-all},audio/esc50/3-128160-A-44.wav,engine
664,{esc50-all},audio/esc50/2-59241-A-35.wav,washing_machine
1062,{esc50-all},audio/esc50/3-159347-B-36.wav,vacuum_cleaner


In [56]:
do_process("esc50-all", ["instruction", "path"])

In [57]:
train_dfs["esc50-all"].sample(5)

,label,file_name,local_path
829,breathing,3-108160-A-23.wav,audio/esc50-all/3-108160-A-23.wav
716,toilet_flush,2-74977-A-18.wav,audio/esc50-all/2-74977-A-18.wav
24,chainsaw,1-116765-A-41.wav,audio/esc50-all/1-116765-A-41.wav
821,crackling_fire,3-104632-A-12.wav,audio/esc50-all/3-104632-A-12.wav
193,crow,1-39835-A-9.wav,audio/esc50-all/1-39835-A-9.wav


## Speech-commands

In [58]:
train_dfs["speech-commands"].sample(5)

,instruction,path,label
26312,{speech-commands-classification},audio/speech_commands/happy/31601aad_nohash_2.wav,happy
27044,{speech-commands-classification},audio/speech_commands/happy/a5609cce_nohash_0.wav,happy
51424,{speech-commands-classification},audio/speech_commands/right/2da58b32_nohash_2.wav,right
46410,{speech-commands-classification},audio/speech_commands/on/94d370bf_nohash_1.wav,on
18464,{speech-commands-classification},audio/speech_commands/follow/d264f7b6_nohash_3...,follow


In [59]:
def do_process_speech_commands(dsname, cols_to_drop=["Unnamed: 0", "path"]):
    
    train_dfs[dsname]["file_name"] = train_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    train_dfs[dsname]["local_path"] = train_dfs[dsname]["path"].copy()
    
    test_dfs[dsname]["file_name"] = test_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    test_dfs[dsname]["local_path"] = test_dfs[dsname]["path"].copy()
    
    validation_dfs[dsname]["file_name"] = validation_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    validation_dfs[dsname]["local_path"] = validation_dfs[dsname]["path"].copy()

    train_dfs[dsname] = train_dfs[dsname].drop(cols_to_drop, axis=1)
    test_dfs[dsname] = test_dfs[dsname].drop(cols_to_drop, axis=1)
    validation_dfs[dsname] = validation_dfs[dsname].drop(cols_to_drop, axis=1)

In [60]:
do_process_speech_commands("speech-commands", ["instruction", "path"])

In [61]:
train_dfs["speech-commands"].sample(5)

,label,file_name,local_path
28677,house,97e0c576_nohash_0.wav,audio/speech_commands/house/97e0c576_nohash_0.wav
47426,on,e5c48e53_nohash_1.wav,audio/speech_commands/on/e5c48e53_nohash_1.wav
61626,six,f2e9b610_nohash_2.wav,audio/speech_commands/six/f2e9b610_nohash_2.wav
55509,seven,80c17118_nohash_0.wav,audio/speech_commands/seven/80c17118_nohash_0.wav
40086,no,7f581e94_nohash_0.wav,audio/speech_commands/no/7f581e94_nohash_0.wav


# Add a 'labels_as_list' column

In [69]:
import pdb

In [72]:
# import ast

# def process_labels(label):
#     if pd.isna(label):
#         return []
    
#     # Convert to string first
#     label_str = str(label)
    
#     # Try to evaluate as literal (handles cases like "[1, 2, 3]" or "['a', 'b']")
#     try:
#         result = ast.literal_eval(label_str)
#         if isinstance(result, list):
#             return result
#         else:
#             return [result]  # Single item
#     except (ValueError, SyntaxError):
#         # Handle comma-separated strings
#         if ',' in label_str:
#             # Split by comma and clean up
#             items = [item.strip() for item in label_str.split(',')]
#             # Try to convert to int if possible
#             processed_items = []
#             for item in items:
#                 try:
#                     processed_items.append(int(item))
#                 except ValueError:
#                     processed_items.append(item)
#             return processed_items
#         else:
#             # Single item
#             try:
#                 return [int(label_str)]
#             except ValueError:
#                 return [label_str]


def process_labels(label):
    if pd.isna(label):
        return []

    if isinstance(label, int):
        return [label]

    if isinstance(label, str):
        if label.lower() == "none":
            return []
            
        items = [item.strip() for item in label.split(',')]
        return items

    else:
        raise ValueError(f"Wrong type, got {type(label)}")

In [ ]:
dd 

In [73]:
for k, df in train_dfs.items():

    if "label" in df:
        df["labels_as_list"] = df["label"].apply(process_labels)
    elif "answer" in df:
        df["labels_as_list"] = df["answer"].apply(process_labels)

    train_dfs[k] = df

In [75]:
for k, df in validation_dfs.items():

    if "label" in df:
        df["labels_as_list"] = df["label"].apply(process_labels)
    elif "answer" in df:
        df["labels_as_list"] = df["answer"].apply(process_labels)

    validation_dfs[k] = df

In [76]:
for k, df in test_dfs.items():

    if "label" in df:
        df["labels_as_list"] = df["label"].apply(process_labels)
    elif "answer" in df:
        df["labels_as_list"] = df["answer"].apply(process_labels)

    test_dfs[k] = df

# Save all as individual jsonls

In [77]:
train_dfs.keys()

dict_keys(['cbi', 'watkins', 'humbugdb', 'egyptian_fruit_bats', 'dogs', 'hiceas', 'dcase', 'enabirds', 'rfcx', 'hainan_gibbons', 'esc50-all', 'speech-commands'])

### check that the new list column labels_as_list has the correct type after saving and loading

In [79]:
train_dfs["rfcx"].to_json(str("rfcx_train.jsonl"), lines=True, orient="records")

In [80]:
rfcx_train = pd.read_json("rfcx_train.jsonl", lines=True, orient="records")

## Save all

In [78]:
beans_root

GSPath('gs://esp-ml-datasets/beans/v0.1.0/raw/')

In [83]:
train_dfs["cbi"].to_json(str(beans_root / "cbi_train.jsonl"), lines=True, orient="records")
train_dfs["watkins"].to_json(str(beans_root / "watkins_train.jsonl"), lines=True, orient="records")
train_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_train.jsonl"), lines=True, orient="records")
train_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_train.jsonl"), lines=True, orient="records")
train_dfs["dogs"].to_json(str(beans_root / "dogs_train.jsonl"), lines=True, orient="records")
train_dfs["hiceas"].to_json(str(beans_root / "hiceas_train.jsonl"), lines=True, orient="records")
train_dfs["dcase"].to_json(str(beans_root / "dcase_train.jsonl"), lines=True, orient="records")
train_dfs["enabirds"].to_json(str(beans_root / "enabirds_train.jsonl"), lines=True, orient="records")
train_dfs["rfcx"].to_json(str(beans_root / "rfcx_train.jsonl"), lines=True, orient="records")
train_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_train.jsonl"), lines=True, orient="records")
train_dfs["esc50-all"].to_json(str(beans_root / "esc50_train.jsonl"), lines=True, orient="records")
train_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_train.jsonl"), lines=True, orient="records")

In [84]:
test_dfs["cbi"].to_json(str(beans_root / "cbi_test.jsonl"), lines=True, orient="records")
test_dfs["watkins"].to_json(str(beans_root / "watkins_test.jsonl"), lines=True, orient="records")
test_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_test.jsonl"), lines=True, orient="records")
test_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_test.jsonl"), lines=True, orient="records")
test_dfs["dogs"].to_json(str(beans_root / "dogs_test.jsonl"), lines=True, orient="records")
test_dfs["hiceas"].to_json(str(beans_root / "hiceas_test.jsonl"), lines=True, orient="records")
test_dfs["dcase"].to_json(str(beans_root / "dcase_test.jsonl"), lines=True, orient="records")
test_dfs["enabirds"].to_json(str(beans_root / "enabirds_test.jsonl"), lines=True, orient="records")
test_dfs["rfcx"].to_json(str(beans_root / "rfcx_test.jsonl"), lines=True, orient="records")
test_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_test.jsonl"), lines=True, orient="records")
test_dfs["esc50-all"].to_json(str(beans_root / "esc50_test.jsonl"), lines=True, orient="records")
test_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_test.jsonl"), lines=True, orient="records")

In [85]:
validation_dfs["cbi"].to_json(str(beans_root / "cbi_val.jsonl"), lines=True, orient="records")
validation_dfs["watkins"].to_json(str(beans_root / "watkins_val.jsonl"), lines=True, orient="records")
validation_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_val.jsonl"), lines=True, orient="records")
validation_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_val.jsonl"), lines=True, orient="records")
validation_dfs["dogs"].to_json(str(beans_root / "dogs_val.jsonl"), lines=True, orient="records")
validation_dfs["hiceas"].to_json(str(beans_root / "hiceas_val.jsonl"), lines=True, orient="records")
validation_dfs["dcase"].to_json(str(beans_root / "dcase_val.jsonl"), lines=True, orient="records")
validation_dfs["enabirds"].to_json(str(beans_root / "enabirds_val.jsonl"), lines=True, orient="records")
validation_dfs["rfcx"].to_json(str(beans_root / "rfcx_val.jsonl"), lines=True, orient="records")
validation_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_val.jsonl"), lines=True, orient="records")
validation_dfs["esc50-all"].to_json(str(beans_root / "esc50_val.jsonl"), lines=True, orient="records")
validation_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_val.jsonl"), lines=True, orient="records")

# Combine all train, test and val

In [86]:
full_train_df = []

for k, df in train_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_train_df.append(df)

In [87]:
full_train_df = pd.concat(full_train_df)
full_train_df.shape

(231819, 5)

In [88]:
full_train_df.sample(10)

,label,file_name,local_path,labels_as_list,source_dataset
38904,None,BUK4_20161011_000804.017_023.wav,audio/dcase/BUK4_20161011_000804.017_023.wav,[],dcase
70360,two,559bc36a_nohash_0.wav,audio/speech_commands/two/559bc36a_nohash_0.wav,[two],speech_commands
21560,four,890cc926_nohash_3.wav,audio/speech_commands/four/890cc926_nohash_3.wav,[four],speech_commands
2040,Ovenbird,Recording_1_Segment_35.000_034.wav,audio/enabirds/Recording_1_Segment_35.000_034.wav,[Ovenbird],enabirds
9209,None,HGSM3AC_0+1_20160312_055400.471_016.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160312_0554...,[],hainan_gibbons
19691,None,HGSM3BD_0+1_20160305_060000.117_000.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160305_0600...,[],hainan_gibbons
20617,American Redstart,2015-09-25_04-00-00_unit10.042_026.wav,audio/dcase/2015-09-25_04-00-00_unit10.042_026...,[American Redstart],dcase
1392,bkchum,XC132348.wav,audio/cbi/wav/XC132348.wav,[bkchum],cbi
13529,"Eastern Towhee, Tufted Titmouse",Recording_4_Segment_26.001_018.wav,audio/enabirds/Recording_4_Segment_26.001_018.wav,"[Eastern Towhee, Tufted Titmouse]",enabirds
81490,yes,f839238a_nohash_0.wav,audio/speech_commands/yes/f839238a_nohash_0.wav,[yes],speech_commands


In [93]:
full_train_df[full_train_df["label"] == "Eastern Towhee, Tufted Titmouse"]["labels_as_list"].iloc[0]

['Eastern Towhee', 'Tufted Titmouse']

In [100]:
full_train_df["label"].dtype

string[python]

In [101]:
full_train_df["labels_as_list"].dtype

dtype('O')

In [95]:
full_train_df["label"] = full_train_df["label"].astype("string")

In [96]:
full_train_df.to_csv("beans_train.csv", index=False)

### read back and check types

In [99]:
ftdf = pd.read_csv("beans_train.csv")
ftdf.sample(5).labels_as_list.iloc[0]

"['learn']"

#### This means it would be better to save as jsonl

In [102]:
full_train_df.to_csv(str(beans_root / "beans_train.csv"), index=False)

In [103]:
full_train_df.to_json(str(beans_root / "beans_train.jsonl"), lines=True, orient="records")

In [109]:
ftdf = pd.read_json(str(beans_root / "beans_train.jsonl"), lines=True, orient="records")
ftdf.sample(5).labels_as_list.iloc[0]

['nine']

In [104]:
full_test_df = []

for k, df in test_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_test_df.append(df)

full_test_df = pd.concat(full_test_df)
full_test_df["label"] = full_test_df["label"].astype("string")

In [105]:
full_test_df.shape

(68044, 5)

In [106]:
full_test_df.to_csv(str(beans_root / "beans_test.csv"), index=False)
full_test_df.to_json(str(beans_root / "beans_test.jsonl"), lines=True, orient="records")

In [107]:
full_val_df = []

for k, df in validation_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_val_df.append(df)

full_val_df = pd.concat(full_val_df)
full_val_df["label"] = full_val_df["label"].astype("string")

In [108]:
full_val_df.to_csv(str(beans_root / "beans_val.csv"), index=False)
full_val_df.to_json(str(beans_root / "beans_val.jsonl"), lines=True, orient="records")